In [1]:
! pip install gensim

In [2]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
print(word_tokenize("Hello world."))   

['Hello', 'world', '.']


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
from google.colab import drive
import sys

# 1. Connect to your Google Drive
drive.mount('/content/drive')

# 2. Add your project folder to the Python path
# (Change 'MyProject' to whatever you named your folder)
sys.path.append('/content/drive/MyDrive/RNN_Project')

# 3. Now your imports will work perfectly!
from Preprocessing import Datacleaner
from visualize import Visualizer
from gensim.models import KeyedVectors

import pandas as pd
import numpy as np 

import re
import html 
import string


from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
df = pd.read_csv(r"/content/drive/MyDrive/RNN_Project/IMDB Dataset.csv")
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


Preprocessing


In [5]:
Datacleaner.remove_duplicated(df)

df['review'] = df['review'].apply(Datacleaner.lower_string)

df['review'] = df['review'].apply(Datacleaner.remove_html)

df['review'] = df['review'].apply(Datacleaner.remove_punctuation)

Datacleaner.remove_duplicated(df)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative


Tokenization and Stopwords removal


In [6]:
df['tokenize'] = df['review'].apply(Datacleaner.tokenize_text)

df['stopwords'] = df['tokenize'].apply(Datacleaner.remove_stopwords)

df['clean_text'] = df['stopwords'].apply(lambda x: ' '.join(x) if x else '')


Mapping Of Sentiment

In [7]:
df['sentiment'] = df['sentiment'].map({
    "positive" : 1, 
    "negative" : 0
})

In [8]:
reviews = df['clean_text'].values
labels = df['sentiment'].values

In [9]:
vocal_size = 20000
tokenize = Tokenizer(num_words=vocal_size, oov_token="")
tokenize.fit_on_texts(reviews)

In [10]:
sequence = tokenize.texts_to_sequences(reviews)
max_length = 200

padded_sequence = pad_sequences(sequence, maxlen=max_length,padding='post',truncating='post')
X_train,X_test,y_train,y_test = train_test_split(padded_sequence,labels,test_size=0.2,random_state=42)

In [11]:
import gensim.downloader as api
word2vec = api.load("word2vec-google-news-300")

In [12]:
#word2vec_path = r'C:\Users\shiva\gensim-data\word2vec-google-news-300\word2vec-google-news-300.gz'
#word2vec = KeyedVectors.load_word2vec_format(model,binary=True)

Creating Embedding Matrix

In [13]:
word_index = tokenize.word_index
embedding_dim = 300
embedding_matrix = np.zeros((vocal_size,embedding_dim))

for word,i in word_index.items():
    if i < vocal_size:
        if word in word2vec:
            embedding_matrix[i] = word2vec[word]

In [14]:
model = Sequential([
    Embedding(input_dim=vocal_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_length,
              trainable=False),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [15]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [16]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     6,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,000,000 (22.89 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 6,000,000 (22.89 MB)

Train Model

In [17]:
history = model.fit(X_train,y_train,epochs=10,batch_size=128,validation_data=(X_test,y_test))


Epoch 1/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 213s 664ms/step - accuracy: 0.5729 - loss: 0.6762 - val_accuracy: 0.5097 - val_loss: 0.6924
Epoch 2/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 208s 670ms/step - accuracy: 0.5169 - loss: 0.6922 - val_accuracy: 0.5141 - val_loss: 0.6899
Epoch 3/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 204s 659ms/step - accuracy: 0.5304 - loss: 0.6854 - val_accuracy: 0.7000 - val_loss: 0.6447
Epoch 4/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 203s 655ms/step - accuracy: 0.5439 - loss: 0.6795 - val_accuracy: 0.5701 - val_loss: 0.6631
Epoch 5/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 205s 660ms/step - accuracy: 0.5343 - loss: 0.6823 - val_accuracy: 0.5227 - val_loss: 0.6887
Epoch 6/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 205s 662ms/step - accuracy: 0.5266 - loss: 0.6893 - val_accuracy: 0.5145 - val_loss: 0.6916
Epoch 7/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 204s 657ms/step - accuracy: 0.5257 - loss: 0.6872 - val_accuracy: 0.5197 - val_loss: 0.6889
Epoch 8/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 205s 663ms/step - accuracy: 0.5464 -